# Projet final : Analyse et visualisation de donnees

L'objectif de ce projet est de realiser une etude complete, de l'exploration des donnees brutes jusqu'a la mise en ligne d'un outil de visualisation interactif.

Ce notebook suit la chaine de traitement de donnees vue en cours et reste dans le cadre du cours : `pandas`, `matplotlib`, `seaborn`, `plotly express`.

Les missions a realiser dans ce notebook sont :
- exploration des donnees brutes,
- chargement et inspection des donnees,
- nettoyage,
- manipulation et preparation des variables,
- analyse graphique avec des visualisations statiques et interactives.

In [ ]:
# importer les packages vus en cours
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

In [ ]:
# dans certains environnements, plotly s'affiche mieux dans le navigateur
pio.renderers.default = 'browser'

## 1. Exploration et Preparation (Pandas)

Objectif : Nettoyer et structurer le jeu de donnees pour l'analyse.

Premiere etape : chargement et inspection des donnees.

In [ ]:
# importer les donnees
df = pd.read_csv('Dataset.csv')

# afficher un apercu du dataset
df.head()

In [ ]:
# verifier la dimension des donnees : nombre de lignes et de colonnes
df.shape

In [ ]:
# afficher les infos des donnees : types des variables et valeurs non nulles
df.info()

In [ ]:
# verifier les valeurs manquantes avant la suite de l'analyse
df.isnull().sum()

## 2. Nettoyage, manipulation et preparation des variables

Dans cette partie, je nettoie et je structure le jeu de donnees pour l'analyse.

In [ ]:
# convertir la colonne de temps en format datetime
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# verifier le resultat apres conversion
df.head()

In [ ]:
# manipulation et preparation des variables temporelles
df['Date'] = df['TransactionStartTime'].dt.date
df['Hour'] = df['TransactionStartTime'].dt.hour
df['Day'] = df['TransactionStartTime'].dt.day
df['Month'] = df['TransactionStartTime'].dt.month

df.head()

In [ ]:
# preparation de variables utiles pour l'analyse financiere
df['AmountAbs'] = df['Amount'].abs()
df['MargeBrute'] = df['Value'] - df['AmountAbs']
df['TauxRentabilite'] = (df['MargeBrute'] / df['AmountAbs'].replace(0, pd.NA)) * 100

df[['Amount', 'AmountAbs', 'Value', 'MargeBrute', 'TauxRentabilite']].head()

## 3. Analyse avec Pandas

Cette partie permet d'obtenir des indicateurs simples avant l'analyse graphique.

In [ ]:
# compter le nombre de transactions dans chaque categorie
df['ProductCategory'].value_counts()

In [ ]:
# compter le nombre de transactions par canal
df['ChannelId'].value_counts()

In [ ]:
# compter le nombre de transactions frauduleuses et non frauduleuses
df['FraudResult'].value_counts()

In [ ]:
# afficher les statistiques descriptives des principales variables quantitatives
df[['Amount', 'Value', 'AmountAbs', 'MargeBrute', 'TauxRentabilite']].describe()

In [ ]:
# calculer le taux de fraude par categorie
fraud_rate_category = df.groupby('ProductCategory')['FraudResult'].mean().sort_values(ascending=False)
fraud_rate_category

In [ ]:
# calculer la valeur moyenne par categorie
df.groupby('ProductCategory')['Value'].mean().sort_values(ascending=False)

In [ ]:
# resumer les montants par strategie de prix
df.groupby('PricingStrategy')[['AmountAbs', 'Value', 'MargeBrute']].agg(['mean', 'sum', 'count']).round(2)

## 4. Analyse Graphique

Objectif : Identifier des tendances et des correlations visuelles.

Cette section combine :
- Statique : utilisation de Matplotlib et Seaborn pour les analyses de fond.
- Interactif : utilisation de Plotly Express pour une exploration dynamique des donnees.

In [ ]:
# calculer le nombre de transactions par jour
transactions_per_day = df.groupby('Date').size()

In [ ]:
# graphique statique : evolution du nombre de transactions par jour
plt.figure(figsize=(14, 5))
transactions_per_day.plot()
plt.title('Nombre de transactions par jour')
plt.xlabel('Date')
plt.ylabel('Nombre de transactions')
plt.show()

In [ ]:
# graphique statique : repartition des transactions par heure
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Hour', color='skyblue')
plt.title('Repartition des transactions par heure')
plt.xlabel('Heure')
plt.ylabel('Nombre de transactions')
plt.show()

In [ ]:
# preparer les donnees du graphique interactif par canal
channel_counts = df['ChannelId'].value_counts().reset_index()
channel_counts.columns = ['ChannelId', 'Count']

In [ ]:
# graphique interactif : repartition des transactions par canal
fig = px.pie(channel_counts, names='ChannelId', values='Count',
             title='Repartition des transactions par canal')
fig.show(renderer='browser')

In [ ]:
# calculer le taux de rentabilite moyen par categorie
rentabilite = df.groupby('ProductCategory')['TauxRentabilite'].mean().sort_values()

In [ ]:
# graphique statique : taux de rentabilite moyen par categorie
plt.figure(figsize=(12, 5))
rentabilite.plot(kind='barh', color='teal')
plt.title('Taux de rentabilite moyen par categorie')
plt.xlabel('Taux de rentabilite')
plt.show()

In [ ]:
# graphique statique : heatmap des correlations
plt.figure(figsize=(8, 5))
sns.heatmap(df[['Amount', 'Value', 'MargeBrute', 'TauxRentabilite', 'FraudResult']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlations entre variables')
plt.show()

In [ ]:
# calculer le revenu total par categorie
revenue_per_category = df.groupby('ProductCategory')['Value'].sum().sort_values(ascending=False).reset_index()

In [ ]:
# graphique interactif : revenu total par categorie de produit
fig = px.bar(revenue_per_category,
             x='ProductCategory', y='Value',
             title='Revenu total par categorie de produit',
             color='Value')
fig.show(renderer='browser')

In [ ]:
# graphique interactif : taux de fraude par categorie
fraud_plot = df.groupby('ProductCategory')['FraudResult'].mean().sort_values(ascending=False).reset_index()
fraud_plot['FraudResult'] = fraud_plot['FraudResult'] * 100

fig = px.bar(fraud_plot, x='ProductCategory', y='FraudResult', color='FraudResult',
             title='Taux de fraude par categorie (%)')
fig.show(renderer='browser')

## 5. Conclusion

Cette etude complete va de l'exploration des donnees brutes jusqu'a la preparation d'un outil de visualisation interactif.

En resumant l'analyse :
- le dataset contient tres peu de fraudes,
- la categorie `financial_services` domine en volume,
- `ChannelId_3` et `ChannelId_2` concentrent l'essentiel des transactions,
- l'analyse graphique permet d'identifier des tendances et des correlations visuelles selon le temps, le canal et la categorie.

Le notebook peut etre complete par un Dashboard Interactif (Streamlit) avec integration de filtres et de graphiques interactifs.